In [0]:
# ======================================
# Dataset: olist_geolocation
# Layer: Silver
# Source: Bronze (Delta)
# Obj: zip_code_prefix + lat + lng
# ======================================

In [0]:
%run ../commons/silver_utils


In [0]:
%run ../../00_config/paths

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType
from pyspark.sql.window import Window

In [0]:
df_bronze = spark.read.format("delta").load(BRONZE_GEOLOCATION_PATH)

In [0]:
display(df_bronze)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("geolocation_lat").cast(DoubleType()).alias("geolocation_lat"),
    F.col("geolocation_lng").cast(DoubleType()).alias("geolocation_lng"),
    F.col("geolocation_zip_code_prefix").cast(StringType()).alias("geolocation_zip_code_prefix"),
    F.trim(F.lower(F.col("geolocation_city"))).alias("geolocation_city"),
    F.upper(F.col("geolocation_state")).alias("geolocation_state")
)

In [0]:
df = df.filter(
    F.col("geolocation_zip_code_prefix").isNotNull() &
    F.col("geolocation_lat").isNotNull() &
    F.col("geolocation_lng").isNotNull()
)

In [0]:
df = df.filter(
    (F.col("geolocation_lat").between(-90, 90)) &
    (F.col("geolocation_lng").between(-180, 180))
)


In [0]:
window = Window.partitionBy(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng"
).orderBy(F.col("geolocation_city"))

df = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())


In [0]:
df.printSchema()

In [0]:
write_silver(df, SILVER_GEOLOCATION_PATH)